[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/08_rmsnorm_solution.ipynb)

# 🟡 Solution: Implement RMSNorm

*Core Ops & Layers · Medium*

Reference implementation. Try it yourself in `08_rmsnorm.ipynb` first.

---
Implement **RMSNorm**, the normalization used by LLaMA, Gemma and most modern LLMs.

$$y = \frac{x}{\sqrt{\frac{1}{D}\sum_i x_i^2 + \epsilon}} \odot w$$

### Signature
```python
def rms_norm(x, weight, eps=1e-6):
    ...
```

### Rules
- Do **not** use `nnx.RMSNorm`
- Normalise over the **last** axis
- **No mean subtraction** and **no bias** term — this is the whole point
- `eps` goes inside the sqrt, and defaults to `1e-6` (not `1e-5`)

### RMSNorm vs LayerNorm
LayerNorm centres *and* scales: it subtracts $\mu$ and divides by $\sigma$.
RMSNorm only scales, dividing by the root-mean-square. So it drops the mean
subtraction and the $\beta$ shift, leaving one learnable vector instead of two.

That turns out to cost nothing in quality while halving the reductions:
LayerNorm needs two passes over the feature axis (the mean, then the variance
around it), RMSNorm needs one (the mean of squares). At LLM scale normalization
is bandwidth-bound rather than FLOP-bound, so dropping a pass over the data is
a real win. This is why essentially every model after LLaMA switched.

### The trap
If your implementation still matches LayerNorm on zero-mean input, that proves
nothing — the two agree exactly when $\mu = 0$. The distinguishing test is
input with a large **non-zero mean**: LayerNorm centres it away, RMSNorm does
not. The tests below use exactly that.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def rms_norm(x, weight, eps=1e-6):
    # Root of the MEAN of the squares — no centring, so no mean subtraction.
    rms = jnp.sqrt(jnp.mean(x ** 2, axis=-1, keepdims=True) + eps)
    return x / rms * weight

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

x = jax.random.normal(jax.random.key(0), (2, 8)) + 10.0   # note the big offset
w = jnp.ones(8)

out = rms_norm(x, w)
print("input mean :", float(x.mean()), "(far from 0)")
print("output mean:", float(out.mean()), "(still far from 0 — RMSNorm does NOT centre)")
print("output RMS :", float(jnp.sqrt(jnp.mean(out ** 2, axis=-1)).mean()), "(~1)")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("rmsnorm")